# 00. Preparar bases — bronze Censo + CPF

Importa **censo_pessoas** + CPF bronze, filtra UF, infere nome da mãe **só no subset UF**, enriquece CEP via `data_cep_uniq.csv`, empilha `registro_unificado` e gera ground truth.

`REBUILD=False` reutiliza tabelas já materializadas no DuckDB.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

REBUILD = False  # True = apaga e reconstrói tudo

from config import (
    CENSO_CEP_ARQUIVO, CENSO_PESSOAS_ARQUIVO, CPF_ARQUIVO,
    COHORT_DEDUP_ARQUIVO, FILTRO_UF, OUTPUT_DIR, USE_PHONETIC_STRIP_VOWELS,
    CENSO_COL_ID_DOMICILIO, CENSO_COL_ID_MORADOR, CENSO_COL_PRIMEIRO_NOME,
    CENSO_COL_SOBRENOME, CPF_COL_CEP, CPF_COL_CPF,
    CPF_COL_DATA_NASC, CPF_COL_NOME, CPF_COL_NOME_MAE, CPF_COL_SEXO,
    benchmark_checkpoint, censo_cep_join_on, censo_dob_sql,
    cep_norm_sql, cpf_norm_sql, cpf_uf_expr, censo_uf_expr,
    export_parquet, get_connection, list_tables, materialize_censo_cep_lookup,
    normalize_date_sql, print_paths, require_input, require_tables,
    uf_filter_clause,
)
from features import featurize_three_part_names_batch, normalize_text, normalize_sexo
from inferir_pais import inferir_nome_mae_duckdb

print_paths()
for label, p in [
    ('CPF', CPF_ARQUIVO), ('CENSO_PESSOAS', CENSO_PESSOAS_ARQUIVO),
    ('CENSO_CEP', CENSO_CEP_ARQUIVO), ('COHORT', COHORT_DEDUP_ARQUIVO),
]:
    require_input(p, label=label)

con = get_connection()
print('REBUILD:', REBUILD, '| FILTRO_UF:', FILTRO_UF)

if not REBUILD and 'registro_unificado' in list_tables(con):
    require_tables(con, ['registro_unificado', 'ground_truth_clusters'], notebook_origem='00')
    print('Tabelas finais já existem — defina REBUILD=True para refazer.')
else:
    print('Prosseguir com pipeline completo nas células abaixo.')


## 1. Inspecionar bronze


In [ ]:
if REBUILD:
    for label, path in [
        ('cpf', CPF_ARQUIVO),
        ('censo_pessoas', CENSO_PESSOAS_ARQUIVO),
    ]:
        print(f'\n=== {label} ===')
        display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}') LIMIT 0").df())


## 2. Importar bronze


In [ ]:
if REBUILD:
    for tbl in [
        'cpf_bronze_raw', 'censo_pessoas_raw', 'cohort_dedup_raw',
        'cpf_filtrado', 'censo_pessoas_filtrado',
        'censo_pais_inferidos', 'censo_cep_lookup', 'censo_morador_cep',
        'cpf_staging', 'cpf_feat', 'cpf_registros',
        'censo_staging', 'censo_feat', 'censo_registros',
        'registro_unificado', 'ground_truth_pairs', 'ground_truth_clusters',
    ]:
        con.execute(f'DROP TABLE IF EXISTS {tbl}')

    con.execute(f"CREATE OR REPLACE TABLE cpf_bronze_raw AS SELECT * FROM read_parquet('{CPF_ARQUIVO}')")
    con.execute(f"CREATE OR REPLACE TABLE censo_pessoas_raw AS SELECT * FROM read_parquet('{CENSO_PESSOAS_ARQUIVO}')")
    con.execute(f"CREATE OR REPLACE TABLE cohort_dedup_raw AS SELECT * FROM read_parquet('{COHORT_DEDUP_ARQUIVO}')")

    for t in ['cpf_bronze_raw', 'censo_pessoas_raw']:
        benchmark_checkpoint(con, t, f'SELECT COUNT(*) FROM {t}')


## 3. Filtrar por UF


In [ ]:
if REBUILD:
    CPF_UF = cpf_uf_expr('c')
    CENSO_UF = censo_uf_expr('p')

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_filtrado AS
    SELECT c.* FROM cpf_bronze_raw c
    WHERE {uf_filter_clause(CPF_UF, FILTRO_UF)}
    ''')

    con.execute(f'''
    CREATE OR REPLACE TABLE censo_pessoas_filtrado AS
    SELECT p.* FROM censo_pessoas_raw p
    WHERE {uf_filter_clause(CENSO_UF, FILTRO_UF)}
    ''')

    for t in ['cpf_filtrado', 'censo_pessoas_filtrado']:
        benchmark_checkpoint(con, t, f'SELECT COUNT(*) FROM {t}')


## 4. Inferir nome da mãe (Censo — **após filtro UF**)

Usa `censo_pessoas_filtrado` — não roda na base nacional inteira.


In [ ]:
if REBUILD:
    inferir_nome_mae_duckdb(con, source_table='censo_pessoas_filtrado')
    benchmark_checkpoint(con, 'censo_pais_inferidos', 'SELECT COUNT(*) FROM censo_pais_inferidos')
    con.execute('''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN nome_mae IS NOT NULL AND TRIM(nome_mae) <> '' THEN 1 ELSE 0 END) AS com_mae,
        ROUND(100.0 * SUM(CASE WHEN nome_mae IS NOT NULL AND TRIM(nome_mae) <> '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mae
    FROM censo_pais_inferidos
    ''').df()


## 5. CEP Censo (`data_cep_uniq.csv` — **após filtro UF**)

LEFT JOIN em `censo_pessoas_filtrado` por `B0000`↔`COD_SETOR` (15 díg.), `NUM_QUADRA`, `NUM_FACE`.

Lookup de CEP filtrado por UF quando `FILTRO_UF` está definido.


In [ ]:
if REBUILD:
    materialize_censo_cep_lookup(con, filtro_uf=FILTRO_UF)
    benchmark_checkpoint(con, 'censo_cep_lookup', 'SELECT COUNT(*) FROM censo_cep_lookup')

    join_on = censo_cep_join_on('p', 'k')
    con.execute(f'''
    CREATE OR REPLACE TABLE censo_morador_cep AS
    SELECT
        CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) AS person_id_censo,
        COALESCE(k.cep, '') AS cep
    FROM censo_pessoas_filtrado p
    LEFT JOIN censo_cep_lookup k ON {join_on}
    ''')

    con.execute('''
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN cep <> '' THEN 1 ELSE 0 END) AS com_cep,
        ROUND(100.0 * SUM(CASE WHEN cep <> '' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cep
    FROM censo_morador_cep
    ''').df()


## 6. CPF — limpeza e nomes


In [ ]:
if REBUILD:
    CPF_N = cpf_norm_sql(f'c."{CPF_COL_CPF}"')
    DT_NASC = normalize_date_sql(f'c."{CPF_COL_DATA_NASC}"')
    NOME_MAE = f'c."{CPF_COL_NOME_MAE}"'
    SEXO = f'c."{CPF_COL_SEXO}"'
    CEP = cep_norm_sql(f'c."{CPF_COL_CEP}"')
    UF_COL = cpf_uf_expr('c')

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_staging AS
    SELECT
        {CPF_N} AS cpf_norm,
        TRIM(CAST(c."{CPF_COL_NOME}" AS VARCHAR)) AS nome_completo_raw,
        {DT_NASC} AS data_nascimento,
        CAST({NOME_MAE} AS VARCHAR) AS nome_mae_raw,
        CAST({SEXO} AS VARCHAR) AS sexo_raw,
        {CEP} AS cep,
        {UF_COL} AS uf
    FROM cpf_filtrado c
    WHERE {CPF_N} IS NOT NULL
    ''')

    featurize_three_part_names_batch(
        con, source_table='cpf_staging', target_table='cpf_feat', name_col='nome_completo_raw',
    )

    phon_sv_select = '''
        , nome_completo_phon_sv, primeiro_nome_phon_sv, ultimo_nome_phon_sv
    ''' if USE_PHONETIC_STRIP_VOWELS else ''

    con.execute(f'''
    CREATE OR REPLACE TABLE cpf_registros AS
    SELECT
        'cpf_' || cpf_norm AS unique_id, 'cpf' AS origem, cpf_norm,
        nome_completo_norm AS nome_completo, primeiro_nome, nome_meio, ultimo_nome,
        nome_completo_phon, primeiro_nome_phon, ultimo_nome_phon
        {phon_sv_select},
        data_nascimento,
        NULLIF(TRIM(CAST(nome_mae_raw AS VARCHAR)), '') AS nome_mae,
        CAST(sexo_raw AS VARCHAR) AS sexo_raw,
        cep, CAST(uf AS VARCHAR) AS uf,
        CAST(NULL AS VARCHAR) AS person_id_censo,
        CAST(NULL AS VARCHAR) AS id_domicilio
    FROM cpf_feat
    ''')
    benchmark_checkpoint(con, 'cpf_registros', 'SELECT COUNT(*) FROM cpf_registros')


## 7. Censo — limpeza e nomes


In [ ]:
if REBUILD:
    DT_PESSOA = censo_dob_sql()
    DT_NASC_C = normalize_date_sql(DT_PESSOA)
    NOME_COMPLETO = f"TRIM(COALESCE(CAST(p.{CENSO_COL_PRIMEIRO_NOME} AS VARCHAR), '') || ' ' || COALESCE(CAST(p.{CENSO_COL_SOBRENOME} AS VARCHAR), ''))"
    UF_C = censo_uf_expr('p')

    con.execute(f'''
    CREATE OR REPLACE TABLE censo_staging AS
    SELECT
        CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) AS person_id_censo,
        CAST(p.{CENSO_COL_ID_DOMICILIO} AS VARCHAR) AS id_domicilio,
        {NOME_COMPLETO} AS nome_completo_raw,
        {DT_NASC_C} AS data_nascimento,
        CAST(p.{CENSO_COL_SEXO} AS VARCHAR) AS sexo_raw,
        {UF_C} AS uf,
        COALESCE(e.cep, '') AS cep,
        m.nome_mae AS nome_mae_inferido
    FROM censo_pessoas_filtrado p
    LEFT JOIN censo_morador_cep e ON CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) = e.person_id_censo
    LEFT JOIN censo_pais_inferidos m ON CAST(p.{CENSO_COL_ID_MORADOR} AS VARCHAR) = m.person_id_censo
    ''')

    featurize_three_part_names_batch(
        con, source_table='censo_staging', target_table='censo_feat', name_col='nome_completo_raw',
    )

    phon_sv_select = '''
        , nome_completo_phon_sv, primeiro_nome_phon_sv, ultimo_nome_phon_sv
    ''' if USE_PHONETIC_STRIP_VOWELS else ''

    con.execute(f'''
    CREATE OR REPLACE TABLE censo_registros AS
    SELECT
        'censo_' || person_id_censo AS unique_id, 'censo' AS origem,
        CAST(NULL AS VARCHAR) AS cpf_norm,
        nome_completo_norm AS nome_completo, primeiro_nome, nome_meio, ultimo_nome,
        nome_completo_phon, primeiro_nome_phon, ultimo_nome_phon
        {phon_sv_select},
        data_nascimento,
        NULLIF(TRIM(CAST(nome_mae_inferido AS VARCHAR)), '') AS nome_mae,
        CAST(sexo_raw AS VARCHAR) AS sexo_raw,
        cep, CAST(uf AS VARCHAR) AS uf,
        person_id_censo, id_domicilio
    FROM censo_feat
    WHERE person_id_censo IS NOT NULL
    ''')
    benchmark_checkpoint(con, 'censo_registros', 'SELECT COUNT(*) FROM censo_registros')


## 8. Normalizar sexo + empilhar


In [ ]:
if REBUILD:
    for tbl in ['cpf_registros', 'censo_registros']:
        df = con.execute(f'SELECT * FROM {tbl}').df()
        df['sexo'] = df['sexo_raw'].map(normalize_sexo)
        df['nome_mae'] = df['nome_mae'].map(lambda x: normalize_text(x) if x else '')
        con.register('_tmp', df)
        con.execute(f'CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM _tmp')
        con.unregister('_tmp')

    phon_cols = (
        'nome_completo_phon_sv, primeiro_nome_phon_sv, ultimo_nome_phon_sv,'
        if USE_PHONETIC_STRIP_VOWELS else ''
    )
    base_cols = '''
        unique_id, origem, cpf_norm, nome_completo, primeiro_nome, nome_meio, ultimo_nome,
        nome_completo_phon, primeiro_nome_phon, ultimo_nome_phon,
    '''
    tail_cols = 'data_nascimento, NULLIF(nome_mae, \'\') AS nome_mae, sexo, cep, uf, person_id_censo, id_domicilio'

    con.execute(f'''
    CREATE OR REPLACE TABLE registro_unificado AS
    SELECT {base_cols} {phon_cols} {tail_cols}
    FROM cpf_registros
    UNION ALL
    SELECT {base_cols} {phon_cols} {tail_cols}
    FROM censo_registros
    ''')

    CPF_GT = cpf_norm_sql('CPF_NORM')
    con.execute(f'''
    CREATE OR REPLACE TABLE ground_truth_pairs AS
    SELECT DISTINCT
        'censo_' || CAST(PERSON_ID_CENSO AS VARCHAR) AS unique_id_censo,
        'cpf_' || {CPF_GT} AS unique_id_cpf,
        CAST(PERSON_ID_CENSO AS VARCHAR) AS person_id_censo,
        {CPF_GT} AS cpf_norm
    FROM cohort_dedup_raw
    WHERE PERSON_ID_CENSO IS NOT NULL AND CPF_NORM IS NOT NULL
    ''')

    con.execute('''
    CREATE OR REPLACE TABLE ground_truth_clusters AS
    WITH pairs AS (
        SELECT unique_id_censo, unique_id_cpf,
               'gt_' || person_id_censo || '_' || cpf_norm AS cluster_id
        FROM ground_truth_pairs
    ),
    labeled AS (
        SELECT unique_id_censo AS unique_id, cluster_id FROM pairs
        UNION ALL
        SELECT unique_id_cpf AS unique_id, cluster_id FROM pairs
    )
    SELECT r.unique_id, COALESCE(l.cluster_id, r.unique_id) AS cluster
    FROM registro_unificado r
    LEFT JOIN labeled l ON r.unique_id = l.unique_id
    ''')

    benchmark_checkpoint(con, 'registro_unificado', 'SELECT COUNT(*) FROM registro_unificado')


## 9. Export


In [ ]:
if REBUILD:
    p1 = export_parquet(con, 'registro_unificado', path=OUTPUT_DIR / 'registro_unificado.parquet')
    p2 = export_parquet(con, 'ground_truth_clusters', path=OUTPUT_DIR / 'ground_truth_clusters.parquet')
    print('Exportado:', p1, p2)
con.close()
